# 06 — So sánh dev, khóa CNN và xem biểu diễn

Bốn cấu hình được đối chiếu theo cùng dev. Trọng số thắng được khóa trước khi xem
biểu diễn; PCA và feature map bên dưới chỉ phục vụ mô tả, không thay quy tắc lựa chọn.
Notebook không đọc ảnh test.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA bằng các mức thư mục giới hạn.
from pathlib import Path
import os, sys

kaggle_input = Path('/kaggle/input')
mounts = []
if kaggle_input.is_dir():
    mounts = [p for pattern in ('datasets/*/*', '*')
              for p in kaggle_input.glob(pattern) if p.is_dir()]
project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *mounts,
                       *(child for mount in mounts for child in mount.glob('*') if child.is_dir())])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/CNN/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError('Gắn đúng một Dataset code có src/CNN/pipeline.py; '
                            f'có thể đặt CS231_PROJECT_ROOT. Tìm thấy: {project_matches}')
PROJECT_ROOT = project_matches[0]
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [PROJECT_ROOT / 'DATA', *(p / 'DATA' for p in mounts), *mounts])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir() and (p / 'dev/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError('Gắn đúng Dataset DATA có train/dev; '
                            f'có thể đặt CS231_DATA_ROOT. Tìm thấy: {data_matches}')
DATA_ROOT = data_matches[0]
result_hint = os.environ.get('CS231_CNN_RESULTS_ROOT')
RESULTS_ROOT = (Path(result_hint) if result_hint else
                Path('/kaggle/working/CS231_CNN/results_224') if Path('/kaggle/working').is_dir()
                else PROJECT_ROOT / 'src/CNN/results_224').resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['CS231_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['CS231_DATA_ROOT'] = str(DATA_ROOT)
print('Code:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
from src.CNN.runtime import restore_previous_output
restore_previous_output(RESULTS_ROOT, '05', 'candidates/C04_DEEP_AUG/run_complete.json')


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Image, Markdown
from src.CNN.data import CLASSES, assert_protocol
from src.CNN.runtime import mark_stage
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())


## 1. Tổng hợp checkpoint của bốn cấu hình

Kiểm tra kết quả từng run, tính lại metric từ dự đoán dev và xác nhận epoch chọn
đúng quy tắc. So cấu hình bằng Macro-F1 cao nhất, hòa thì MAE thấp hơn,
vẫn hòa ưu tiên C01…C04 rồi epoch sớm hơn. Lưu cả lịch sử mọi epoch để đối chiếu.


In [ ]:
from src.CNN.pipeline import select_and_lock
comparison, lock = select_and_lock(RESULTS_ROOT, PROJECT_ROOT)
display(comparison)
print('Đã khóa:', lock['candidate'], '| epoch:', lock['best_epoch'])
display(pd.DataFrame([lock['dev_metrics']]))


## 2. Vector GAP và PCA fit trên train

Model.eval() tạo vector GAP128/256 chiều cho toàn bộ train/dev mà không augmentation.
PCA hai chiều fit trên train rồi transform dev; biểu đồ không tham gia bộ phân loại
và không được dùng để đổi cấu hình đã khóa. Nhãn màu chỉ giúp đọc sự phân bố trên hai trục.


In [ ]:
from src.CNN.analysis import inspect_features
coordinates, selected_channels = inspect_features(DATA_ROOT, RESULTS_ROOT, PROJECT_ROOT)
display(coordinates.head())
display(Image(filename=str(RESULTS_ROOT / 'feature_analysis/embedding_pca_dev.png')))


## 3. Trực quan feature map

Một ảnh train cố định được xem tại khối đầu và khối cuối. Mỗi khối lấy sáu kênh có
đáp ứng trung bình lớn nhất. Thang màu từng kênh được co giãn riêng;
không suy rằng mỗi kênh chỉ học một loại dấu hiệu hoặc độ sáng giữa hai ô có cùng độ lớn.


In [ ]:
display(selected_channels)
display(Image(filename=str(RESULTS_ROOT / 'feature_analysis/feature_maps.png')))
display(pd.DataFrame([json.loads((RESULTS_ROOT / 'feature_analysis/analysis_record.json').read_text())]))


## 4. Bàn giao trọng số đã khóa

Khóa ghi SHA-256 của checkpoint, protocol, manifest, lịch sử, dự đoán và code.
Bước07 nạp chính best.pt này để chấm test, không fit lại train hoặc train+dev.
Khi code/dữ liệu/thiết lập thay đổi, cần một lần thực nghiệm mới thay vì sửa hash.


In [ ]:
mark_stage(RESULTS_ROOT, '06')
